# Task 2: Booking data

In [1]:
DATA = "bookings.jsonl"

## Part A: Validate bookings

The models below check the booking data against the task's rules. Invalid bookings go into `quarantine.jsonl` with a reason.

`Decimal` is used for money to avoid floating-point rounding errors. Extra fields are kept. Booking timestamps are converted to UTC, and bookings must be made before the event day.

In [2]:
import json
import shutil
from collections import Counter
from datetime import date as Date, timezone
from decimal import Decimal
from pathlib import Path
from typing import Literal

import duckdb
import pandas as pd
from IPython.display import display
from pydantic import AwareDatetime, BaseModel, ConfigDict, Field, TypeAdapter, ValidationError, model_validator
from pydantic_core import PydanticCustomError


class Ticket(BaseModel):
    model_config = ConfigDict(extra="allow")
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1, strict=True)
    price: Decimal = Field(ge=0, allow_inf_nan=False)


class Student(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str = Field(pattern=r"^[0-9]{2}-[0-9]{4}$")
    name: str
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    title: str
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: Date
    venue: str


class Booking(BaseModel):
    model_config = ConfigDict(extra="allow")
    booking_id: str
    booked_at: AwareDatetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: Decimal = Field(allow_inf_nan=False)
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]

    @model_validator(mode="after")
    def check_total(self):
        if self.total != sum((t.qty * t.price for t in self.tickets), Decimal(0)):
            raise PydanticCustomError("total_mismatch", "total does not equal ticket amounts")
        return self

    @model_validator(mode="after")
    def check_booking_date(self):
        self.booked_at = self.booked_at.astimezone(timezone.utc)
        if self.booked_at.date() >= self.event.date:
            raise PydanticCustomError("booking_date", "booked_at is not before event date")
        return self


def validation_reasons(exc):
    reasons = set()
    for error in exc.errors(include_url=False):
        # Group errors by field, rather than splitting counts by ticket list position.
        field = ".".join(str(p) for p in error["loc"] if not isinstance(p, int))
        reasons.add(f"{field}: {error['msg']}" if field else error["msg"])
    return sorted(reasons)


def process_lines(lines, deduplicate_first=False):
    accepted, rejected, documents, seen = [], [], [], set()
    for line_number, line in enumerate(lines, start=1):
        booking_id = None
        try:
            raw = json.loads(line, parse_float=Decimal)
        except (json.JSONDecodeError, ValueError):
            reasons = ["invalid JSON"]
        else:
            documents.append((line_number, raw))
            booking_id = raw.get("booking_id") if isinstance(raw, dict) else None
            duplicate = isinstance(booking_id, str) and booking_id in seen
            if deduplicate_first and duplicate:
                reasons = ["duplicate booking_id"]
            else:
                if deduplicate_first and isinstance(booking_id, str):
                    seen.add(booking_id)
                try:
                    booking = Booking.model_validate(raw)
                except ValidationError as exc:
                    reasons = validation_reasons(exc)
                else:
                    if not deduplicate_first and booking.booking_id in seen:
                        reasons = ["duplicate booking_id"]
                    else:
                        seen.add(booking.booking_id)
                        accepted.append(booking)
                        continue
        rejected.append({"line_number": line_number, "booking_id": booking_id,
                         "reason": "; ".join(reasons), "reasons": reasons})
    return accepted, rejected, documents


lines = Path(DATA).read_text(encoding="utf-8").splitlines()
valid, quarantine, documents = process_lines(lines)
with Path("quarantine.jsonl").open("w", encoding="utf-8") as stream:
    for entry in quarantine:
        stream.write(json.dumps(entry, ensure_ascii=False, default=str) + "\n")

reason_counts = Counter(reason for entry in quarantine for reason in entry["reasons"])
summary = {"lines_read": len(lines), "passed": len(valid), "quarantined": len(quarantine)}
assert summary["lines_read"] == summary["passed"] + summary["quarantined"]
display(pd.DataFrame([summary]))
display(pd.DataFrame(sorted(reason_counts.items()), columns=["reason", "quarantined_lines"]))
print("A line with several failures is counted once per distinct reported reason.")

,lines_read,passed,quarantined
0,283,259,24


,reason,quarantined_lines
0,booked_at is not before event date,3
1,duplicate booking_id,3
2,event: Input should be a valid dictionary or i...,2
3,student.id: String should match pattern '^[0-9...,4
4,tickets.qty: Input should be greater than or e...,3
5,"tickets.type: Input should be 'standard', 'vip...",3
6,tickets: List should have at least 1 item afte...,2
7,total does not equal ticket amounts,4


A line with several failures is counted once per distinct reported reason.


### Compare validation and duplicate checks

The main pipeline validates each booking before checking for duplicates. This lets a corrected booking use the same ID as an earlier invalid booking.

The next cell compares both orders to see whether the counts change. The validation-first results are used for the rest of the notebook.

In [3]:
valid_first_seen, quarantine_first_seen, _ = process_lines(lines, deduplicate_first=True)
comparison = pd.DataFrame([
    {"order": "validate then deduplicate", "passed": len(valid), "quarantined": len(quarantine)},
    {"order": "deduplicate then validate", "passed": len(valid_first_seen),
     "quarantined": len(quarantine_first_seen)},
])
display(comparison)
before_counts = Counter(r for entry in quarantine_first_seen for r in entry["reasons"])
display(pd.DataFrame([
    {"reason": reason, "validation_first": reason_counts[reason],
     "duplicates_first": before_counts[reason]}
    for reason in sorted(reason_counts.keys() | before_counts.keys())
]))

,order,passed,quarantined
0,validate then deduplicate,259,24
1,deduplicate then validate,259,24


,reason,validation_first,duplicates_first
0,booked_at is not before event date,3,3
1,duplicate booking_id,3,3
2,event: Input should be a valid dictionary or i...,2,2
3,student.id: String should match pattern '^[0-9...,4,4
4,tickets.qty: Input should be greater than or e...,3,3
5,"tickets.type: Input should be 'standard', 'vip...",3,3
6,tickets: List should have at least 1 item afte...,2,2
7,total does not equal ticket amounts,4,4


## Part B: Create flat tables and find new fields

`bookings` has one row per accepted booking, with student and event details in separate columns. `booking_tickets` has one row per ticket entry, including its quantity, price and amount.

The assertions check that booking IDs are unique and each booking's ticket amounts add up to its total.

Schema drift means a field has been added to the data. We check all parsed documents for extra fields and report the earliest valid timestamp and the first file line where each field appears. Missing optional fields are shown as null.

In [4]:
booking_columns = ["booking_id", "booked_at", "total", "payment", "status",
                   "student_id", "student_name", "student_faculty", "event_id",
                   "event_title", "event_club", "event_date", "event_venue"]
bookings = pd.json_normalize(
    [b.model_dump(exclude={"tickets"}) for b in valid], sep="_"
)
bookings = bookings.reindex(columns=list(dict.fromkeys(booking_columns + list(bookings.columns))))
booking_tickets = pd.DataFrame([
    {"booking_id": b.booking_id, "type": t.type, "qty": t.qty,
     "price": t.price, "amount": t.qty * t.price}
    for b in valid for t in b.tickets
], columns=["booking_id", "type", "qty", "price", "amount"])

assert bookings["booking_id"].is_unique
ticket_totals = booking_tickets.groupby("booking_id")["amount"].agg(lambda x: sum(x, Decimal(0)))
assert set(ticket_totals.index) == set(bookings["booking_id"])
assert all(ticket_totals[row.booking_id] == row.total for row in bookings.itertuples())
display(bookings.head())
display(booking_tickets.head())
print(f"{len(bookings)} bookings; {len(booking_tickets)} ticket entries. Assertions passed.")

timestamp_adapter = TypeAdapter(AwareDatetime)


def drift_timestamp(value):
    try:
        timestamp = timestamp_adapter.validate_python(value)
    except ValidationError:
        return pd.NaT
    return pd.to_datetime(timestamp, utc=True, errors="coerce")


# Rejected timestamps must not crash drift detection or determine its onset.
assert pd.isna(drift_timestamp({"date": "broken"}))
assert pd.isna(drift_timestamp(["2026-09-27T12:29:00Z", "2026-09-28T12:29:00Z"]))
assert drift_timestamp("2026-10-20T01:05:00Z") == pd.Timestamp("2026-10-20T01:05:00Z")

expected_fields = {
    "": set(Booking.model_fields), "student": set(Student.model_fields),
    "event": set(Event.model_fields), "tickets": set(Ticket.model_fields),
}
drift_occurrences = []
for line_number, raw in documents:
    if not isinstance(raw, dict):
        continue
    scopes = [("", raw), ("student", raw.get("student")), ("event", raw.get("event"))]
    if isinstance(raw.get("tickets"), list):
        scopes.extend(("tickets", t) for t in raw["tickets"])
    for scope, obj in scopes:
        if not isinstance(obj, dict):
            continue
        for key in obj.keys() - expected_fields[scope]:
            drift_occurrences.append({
                "field": f"{scope}.{key}" if scope else key,
                "line_number": line_number, "booking_id": raw.get("booking_id"),
                "booked_at": drift_timestamp(raw.get("booked_at")),
            })

drift_rows = []
for field in sorted({r["field"] for r in drift_occurrences}):
    occurrences = [r for r in drift_occurrences if r["field"] == field]
    dated = [r for r in occurrences if pd.notna(r["booked_at"])]
    first_line = min(occurrences, key=lambda r: r["line_number"])
    earliest = min(dated, key=lambda r: r["booked_at"]) if dated else None
    drift_rows.append({"field": field, "documents": len({r["line_number"] for r in occurrences}),
                       "earliest_booked_at": earliest["booked_at"] if earliest else pd.NaT,
                       "earliest_booking_id": earliest["booking_id"] if earliest else None,
                       "first_file_line": first_line["line_number"],
                       "first_file_booking_id": first_line["booking_id"]})
drift_report = pd.DataFrame(drift_rows, columns=["field", "documents", "earliest_booked_at",
    "earliest_booking_id", "first_file_line", "first_file_booking_id"])
display(drift_report)

,booking_id,booked_at,total,payment,status,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,promo_code
0,B0001,2026-09-28 15:46:00+00:00,340.0,card,cancelled,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,NaN
1,B0002,2026-09-27 12:29:00+00:00,40.0,card,confirmed,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,NaN
2,B0003,2026-10-13 20:19:00+00:00,60.0,wallet,confirmed,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,NaN
3,B0004,2026-11-01 06:42:00+00:00,105.0,card,confirmed,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,CLUB10
4,B0005,2026-10-28 19:06:00+00:00,20.0,cash,confirmed,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,None


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


259 bookings; 314 ticket entries. Assertions passed.


,field,documents,earliest_booked_at,earliest_booking_id,first_file_line,first_file_booking_id
0,promo_code,193,2026-10-20 01:05:00+00:00,B0125,4,B0004


## Part C: DuckDB

`dim_event` stores event details. `dim_student` stores student details. Each event and student gets a numeric key, assigned in ID order. If the same ID has different details, the details from the lowest accepted booking ID are used.

`fact_ticket_sales` stores one row per ticket entry and uses these keys to link to the event and student tables. It stores ticket quantities, prices and amounts. Repeating a booking's total on every ticket entry would make the total too large when summed.

`booked_date_key` uses YYYYMMDD. `booked_month` uses YYYY-MM. Both use the UTC booking date.

In [5]:
con = duckdb.connect(":memory:")
# Give empty inputs useful types too, so an all-invalid hidden file still runs.
sql_bookings = bookings[booking_columns].copy()
for column in ["booking_id", "payment", "status", "student_id", "student_name",
               "student_faculty", "event_id", "event_title", "event_club", "event_venue"]:
    sql_bookings[column] = sql_bookings[column].astype("string")
sql_bookings["booked_at"] = pd.to_datetime(sql_bookings["booked_at"], utc=True)
sql_bookings["event_date"] = pd.to_datetime(sql_bookings["event_date"])
sql_tickets = booking_tickets.copy()
for column in ["booking_id", "type"]:
    sql_tickets[column] = sql_tickets[column].astype("string")
sql_tickets["qty"] = sql_tickets["qty"].astype("int64")
if bookings.empty:
    sql_bookings["total"] = sql_bookings["total"].astype("float64")
    for column in ["price", "amount"]:
        sql_tickets[column] = sql_tickets[column].astype("float64")
con.register("bookings_input", sql_bookings)
con.register("tickets_input", sql_tickets)
con.execute("""
    CREATE TABLE bookings AS SELECT * FROM bookings_input;
    CREATE TABLE booking_tickets AS SELECT * FROM tickets_input;
    CREATE TABLE dim_event AS
    WITH events AS (
        SELECT event_id, first(event_title ORDER BY booking_id) AS title,
               first(event_club ORDER BY booking_id) AS club,
               first(CAST(event_date AS DATE) ORDER BY booking_id) AS event_date,
               first(event_venue ORDER BY booking_id) AS venue
        FROM bookings GROUP BY event_id
    )
    SELECT row_number() OVER (ORDER BY event_id) AS event_key, * FROM events;

    CREATE TABLE dim_student AS
    WITH students AS (
        SELECT student_id, first(student_name ORDER BY booking_id) AS name,
               first(student_faculty ORDER BY booking_id) AS faculty
        FROM bookings GROUP BY student_id
    )
    SELECT row_number() OVER (ORDER BY student_id) AS student_key, * FROM students;

    CREATE TABLE fact_ticket_sales AS
    SELECT t.booking_id, e.event_key, s.student_key, t.type AS ticket_type,
           t.qty, t.price, t.amount, b.status,
           CAST(strftime(b.booked_at AT TIME ZONE 'UTC', '%Y%m%d') AS INTEGER) AS booked_date_key,
           strftime(b.booked_at AT TIME ZONE 'UTC', '%Y-%m') AS booked_month
    FROM booking_tickets t
    JOIN bookings b USING (booking_id)
    JOIN dim_event e USING (event_id)
    JOIN dim_student s USING (student_id);
""")

fact_count, fact_amount = con.execute(
    "SELECT count(*), coalesce(sum(amount), 0) FROM fact_ticket_sales"
).fetchone()
booking_amount = sum(bookings["total"], Decimal(0))
assert fact_count == len(booking_tickets)
assert fact_amount == booking_amount
assert con.execute("SELECT count(*) = count(DISTINCT event_id) FROM dim_event").fetchone()[0]
assert con.execute("SELECT count(*) = count(DISTINCT student_id) FROM dim_student").fetchone()[0]
print(f"Fact rows: {fact_count}; reconciled amount: EGP {fact_amount}")
display(con.sql("SELECT * FROM dim_event").df())
display(con.sql("SELECT * FROM dim_student LIMIT 5").df())
display(con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").df())

Fact rows: 314; reconciled amount: EGP 31285.0


,event_key,event_id,title,club,event_date,venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,student_key,student_id,name,faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,booked_date_key,booked_month
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928,2026-09
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928,2026-09
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927,2026-09
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013,2026-10
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101,2026-11


### Tickets sold and revenue

For confirmed bookings, group the results by club and ticket type. Add up `qty` for the number of tickets sold and `amount` for revenue.

In [6]:
sales_query = """
    SELECT e.club, f.ticket_type, sum(f.qty) AS tickets_sold, sum(f.amount) AS revenue_egp
    FROM fact_ticket_sales f
    JOIN dim_event e USING (event_key)
    WHERE f.status = 'confirmed'
    GROUP BY e.club, f.ticket_type
    ORDER BY e.club, f.ticket_type
"""
sales_by_club = con.sql(sales_query).df()
display(sales_by_club)
repeated_total = con.execute("""
    SELECT coalesce(sum(b.total), 0)
    FROM fact_ticket_sales f JOIN bookings b USING (booking_id)
""").fetchone()[0]
print(f"Correct booking total: EGP {booking_amount}; "
      f"SUM(total) if repeated in the fact: EGP {repeated_total}")

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45.0,900.0
1,Chess,student,44.0,440.0
2,Debate,standard,65.0,1950.0
3,Debate,student,34.0,510.0
4,Debate,vip,40.0,3000.0
5,Football,standard,64.0,1600.0
6,Music,standard,67.0,2930.0
7,Music,student,20.0,1000.0
8,Music,vip,78.0,9990.0
9,Robotics,standard,60.0,2400.0


Correct booking total: EGP 31285.0; SUM(total) if repeated in the fact: EGP 42000.0


### Save the tables as Parquet

Save each dimension table in one file. Save ticket entries by booking month under `star/fact_ticket_sales/booked_month=YYYY-MM/`.

Each run replaces the generated files and removes old month folders. The files are then read back to check the row counts and total amount. If there are no accepted bookings, save an empty fact file.

In [7]:
star = Path("star")
star.mkdir(exist_ok=True)
for table in ["dim_event", "dim_student"]:
    con.execute(f"COPY {table} TO 'star/{table}.parquet' (FORMAT PARQUET)")
fact_path = star / "fact_ticket_sales"
if fact_path.exists():
    shutil.rmtree(fact_path)
fact_path.mkdir()
if fact_count:
    con.execute("""
        COPY fact_ticket_sales TO 'star/fact_ticket_sales'
        (FORMAT PARQUET, PARTITION_BY (booked_month))
    """)
else:
    con.execute("COPY fact_ticket_sales TO 'star/fact_ticket_sales/empty.parquet' (FORMAT PARQUET)")

for table in ["dim_event", "dim_student"]:
    stored = con.execute(f"SELECT count(*) FROM read_parquet('star/{table}.parquet')").fetchone()[0]
    assert stored == con.execute(f"SELECT count(*) FROM {table}").fetchone()[0]
stored_count, stored_amount = con.execute("""
    SELECT count(*), coalesce(sum(amount), 0)
    FROM read_parquet('star/fact_ticket_sales/**/*.parquet', hive_partitioning=true)
""").fetchone()
assert stored_count == fact_count
assert stored_amount == booking_amount
print("Parquet read-back assertions passed.")
for path in sorted(star.rglob("*.parquet")):
    print(path)

Parquet read-back assertions passed.
star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
